# Pipeline and Column Transformer

Without a pipeline, every preprocessing step must be applied manually in the correct order, on the correct dataset. The risk is applying fit to validation or test data, which leaks information and produces overly optimistic evaluation results.

`sklearn.pipeline.Pipeline` chains a sequence of transformers followed by a final estimator into a single object.

### ColumnTransformer

A single pipeline **applies the same steps to all columns**. When different columns require different transformations, ColumnTransformer handles this by applying separate pipelines to separate column groups in parallel, then concatenating the results horizontally.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   Ridge())
])

pipe.fit(X_train, y_train)    # üç adımı sırayla train'e uygular
pipe.predict(X_val)           # val'e sadece transform uygular, fit etmez

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Sütun grupları
numerical_cols   = ["LotArea", "GrLivArea", "YearBuilt"]
ordinal_cols     = ["Exter Qual", "Kitchen Qual"]
onehot_cols      = ["Central Air", "Street"]

# Her grup için kendi pipeline'ı
numerical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["Po","Fa","TA","Gd","Ex"],
                                           ["Po","Fa","TA","Gd","Ex"]]))
])

onehot_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Hepsini bir araya getir
preprocessor = ColumnTransformer([
    ("num", numerical_pipe, numerical_cols),
    ("ord", ordinal_pipe,   ordinal_cols),
    ("ohe", onehot_pipe,    onehot_cols),
])